<div style="background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 35px; border-radius: 12px; color: white; margin-bottom: 25px;">
  <h1 style="margin: 0 0 8px 0; font-size: 2em;">🏦 Lab 02: Build a MAF Client Advisor Agent</h1>
  <p style="margin: 0; font-size: 1.15em; opacity: 0.92;">Zava Bank Workshop — Hosted Agents with Microsoft Agent Framework</p>
</div>

<div style="background: #f5f5f5; border-left: 4px solid #533483; padding: 16px 20px; border-radius: 6px; margin-bottom: 20px;">

## What We Learn

- Build a custom Zava Bank advisor using MAF's **`BaseAgent`** pattern
- Implement the **`on_message`** handler to process advisory requests
- Use the model for reasoning over portfolio data
- Manage **multi-turn conversations** within the agent class
- Compare the development experience to prompt agents (Path A)

</div>

---
## 1 · Setup

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv

notebook_dir = Path.cwd()
env_path = notebook_dir
while env_path != env_path.parent:
    if (env_path / ".env").exists():
        break
    env_path = env_path.parent

load_dotenv(env_path / ".env", override=True)

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-4o")

print(f"Project endpoint : {PROJECT_ENDPOINT[:60]}..." if len(PROJECT_ENDPOINT) > 60 else f"Project endpoint : {PROJECT_ENDPOINT}")
print(f"Model deployment : {MODEL_DEPLOYMENT}")

In [ ]:
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.agentserver.agentframework import BaseAgent, AgentContext

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)

# Load Zava Bank data
DATA_DIR = notebook_dir.parents[1] / "data" / "zava-bank"
portfolios_df = pd.read_csv(DATA_DIR / "client_portfolios.csv")
market_df     = pd.read_csv(DATA_DIR / "market_data.csv")
risk_df       = pd.read_csv(DATA_DIR / "risk_metrics.csv")

print(f"✅ Clients loaded  — {portfolios_df['client_id'].nunique()} clients, {len(portfolios_df)} positions")
print(f"✅ Market loaded   — {len(market_df)} indicators")
print(f"✅ Risk loaded     — {len(risk_df)} profiles")

---
## 2 · Define the ZavaBankAdvisor

This is the core of the MAF approach. The agent is a Python class — you control what happens when a message arrives.

In [ ]:
class ZavaBankAdvisor(BaseAgent):
    """Zava Bank Client Advisory Agent — MAF implementation."""

    SYSTEM_PROMPT = """You are the Zava Bank Client Advisor, a production-grade financial advisory assistant.

    You help financial advisors by:
    - Analyzing client portfolios and positions
    - Assessing portfolio risk metrics
    - Providing relevant market context

    Important compliance rules:
    - You are an advisor-assist tool — never give advice directly to clients
    - Never recommend specific real stocks or execute trades
    - Always include appropriate disclaimers
    - Stay grounded in the data provided by your tools
    """

    async def on_message(self, context: AgentContext, message: str) -> str:
        """Handle incoming advisory requests."""
        response = await context.invoke_model(
            system_prompt=self.SYSTEM_PROMPT,
            user_message=message,
        )
        return response


print("✅ ZavaBankAdvisor class defined")
print(f"   Base class : {ZavaBankAdvisor.__bases__}")
print(f"   Handler    : on_message")

<div style="background: #f5f5f5; border-left: 4px solid #0f3460; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">

**What just happened?** We defined a Python class that:
1. Inherits from `BaseAgent` — the MAF contract
2. Carries a `SYSTEM_PROMPT` as a class attribute — no external configuration file needed
3. Implements `on_message` — the single method the framework calls when a user message arrives
4. Uses `context.invoke_model()` to send the prompt + message to the underlying LLM

Compare this to Path A, where the system prompt and tool schemas were registered as JSON via the Foundry API. Here, everything is Python.

</div>

---
## 3 · Add Data-Aware Tool Methods

In a full MAF agent, tools are regular Python methods. The agent decides when to call them based on the user's message. Here we add portfolio lookup and risk retrieval as methods on the class.

In [ ]:
class ZavaBankAdvisor(BaseAgent):
    """Zava Bank Client Advisory Agent — MAF implementation with tools."""

    SYSTEM_PROMPT = """You are the Zava Bank Client Advisor, a production-grade financial advisory assistant.

    You help financial advisors by:
    - Analyzing client portfolios and positions
    - Assessing portfolio risk metrics
    - Providing relevant market context

    When the user asks about a client, use the available data to ground your analysis.
    Format monetary values with $ and two decimal places.
    Present tabular data clearly.

    Important compliance rules:
    - You are an advisor-assist tool — never give advice directly to clients
    - Never recommend specific real stocks or execute trades
    - Always include appropriate disclaimers
    - Stay grounded in the data provided by your tools
    """

    def __init__(self, portfolios: pd.DataFrame, risk: pd.DataFrame, market: pd.DataFrame):
        super().__init__()
        self._portfolios = portfolios
        self._risk = risk
        self._market = market

    def lookup_portfolio(self, client_id: str) -> str:
        """Retrieve portfolio positions for a client."""
        rows = self._portfolios[self._portfolios["client_id"] == client_id]
        if rows.empty:
            return f"No portfolio found for client {client_id}."
        return rows.to_json(orient="records", indent=2)

    def lookup_risk(self, client_id: str) -> str:
        """Retrieve risk metrics for a client."""
        rows = self._risk[self._risk["client_id"] == client_id]
        if rows.empty:
            return f"No risk profile found for client {client_id}."
        return rows.to_json(orient="records", indent=2)

    def get_market_context(self) -> str:
        """Return current market indicators."""
        return self._market.to_json(orient="records", indent=2)

    async def on_message(self, context: AgentContext, message: str) -> str:
        """Handle incoming advisory requests with data grounding."""

        # Simple keyword-based tool dispatch — production agents
        # would use the model's function-calling capability here.
        supplemental_data = []

        # Check if the message references a client ID
        for cid in self._portfolios["client_id"].unique():
            if cid.lower() in message.lower():
                supplemental_data.append(f"Portfolio for {cid}:\n{self.lookup_portfolio(cid)}")
                supplemental_data.append(f"Risk metrics for {cid}:\n{self.lookup_risk(cid)}")
                break

        # Check if the message asks about market conditions
        market_keywords = ["market", "index", "indicator", "economy", "trend"]
        if any(kw in message.lower() for kw in market_keywords):
            supplemental_data.append(f"Market indicators:\n{self.get_market_context()}")

        # Build the augmented user message
        augmented_message = message
        if supplemental_data:
            data_block = "\n\n".join(supplemental_data)
            augmented_message = f"{message}\n\n--- Retrieved Data ---\n{data_block}"

        response = await context.invoke_model(
            system_prompt=self.SYSTEM_PROMPT,
            user_message=augmented_message,
        )
        return response


print("✅ ZavaBankAdvisor v2 defined — now with data-aware tool methods")

<div style="background: #f5f5f5; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">

**Key difference from Path A**: In the prompt-agent approach, tool schemas were JSON definitions registered with Foundry, and the platform executed them server-side. Here, tools are Python methods on the agent class. The agent decides when to call them, retrieves the data, and injects it into the model context — all in your process.

</div>

---
## 4 · Register &amp; Test the Agent

In [ ]:
# Instantiate the advisor with our Zava Bank data
advisor = ZavaBankAdvisor(
    portfolios=portfolios_df,
    risk=risk_df,
    market=market_df,
)

print(f"Agent class  : {type(advisor).__name__}")
print(f"Clients known: {portfolios_df['client_id'].nunique()}")
print(f"Tool methods : lookup_portfolio, lookup_risk, get_market_context")

### Test: Simple Advisory Query

Send a message referencing a specific client. The agent should detect the client ID, pull portfolio and risk data, and provide a grounded response.

In [ ]:
# NOTE: This cell requires a running AgentContext from the MAF runtime.
# In a local notebook without the hosted runtime, we demonstrate the
# tool-dispatch logic directly.

test_message = "Give me a portfolio overview for client ZB-CL-001"

# Simulate the tool-dispatch portion of on_message
print(f"📨 User message: {test_message}\n")

portfolio_data = advisor.lookup_portfolio("ZB-CL-001")
risk_data = advisor.lookup_risk("ZB-CL-001")

print("📊 Portfolio data retrieved:")
print(json.dumps(json.loads(portfolio_data)[:3], indent=2))  # Show first 3 positions
print(f"   ... ({len(json.loads(portfolio_data))} total positions)\n")

print("⚠️ Risk metrics retrieved:")
risk_parsed = json.loads(risk_data)
print(json.dumps(risk_parsed, indent=2))

---
## 5 · Multi-Turn Conversation Handling

In MAF, conversation state is managed through `context.session`. Each call to `on_message` can read and write session state. Here's the pattern:

In [ ]:
# Multi-turn pattern — the agent tracks conversation history
# in a list and passes it to the model on each turn.

conversation_history = []

turns = [
    "What is client ZB-CL-001's risk tolerance?",
    "How concentrated is their portfolio?",
    "What market indicators should I watch for this client?",
]

for i, user_msg in enumerate(turns, 1):
    print(f"--- Turn {i} ---")
    print(f"👤 Advisor: {user_msg}")

    conversation_history.append({"role": "user", "content": user_msg})

    # In a live MAF agent, context.invoke_model would handle this.
    # Here we show the data retrieval that would augment each turn.
    if "ZB-CL-001" in user_msg or i > 1:  # Follow-ups reference prior context
        risk_info = advisor.lookup_risk("ZB-CL-001")
        risk_parsed = json.loads(risk_info)[0]
        print(f"🤖 [Tool call] Risk tolerance: {risk_parsed.get('risk_tolerance', 'N/A')}")
        print(f"               Top-3 concentration: {risk_parsed.get('concentration_top3_pct', 'N/A')}%")
        print(f"               Portfolio beta: {risk_parsed.get('portfolio_beta', 'N/A')}")

    conversation_history.append({"role": "assistant", "content": "[response would go here]"})
    print()

print(f"💬 Conversation length: {len(conversation_history)} messages ({len(turns)} turns)")

<div style="background: #f5f5f5; border-left: 4px solid #533483; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">

**Multi-turn in MAF vs Prompt Agents**

| Aspect | Prompt Agent (Path A) | MAF (Path B) |
|---|---|---|
| **State management** | Foundry manages thread history automatically | You manage history in `context.session` or your own data structure |
| **Context window** | Platform handles truncation | You control what goes into each `invoke_model` call |
| **Follow-up logic** | Implicit — model sees full thread | Explicit — you decide what prior context to include |

The trade-off is clear: MAF requires more code, but you get deterministic control over what the model sees at each turn.

</div>

---
## 6 · Path A vs Path B — Side by Side

Now that we've built agents both ways, here's a direct comparison of the developer experience.

In [ ]:
comparison = {
    "Define agent": [
        "agents_client.create_agent(model, instructions, tools)",
        "class ZavaBankAdvisor(BaseAgent): ...",
    ],
    "Add a tool": [
        "FunctionTool with JSON schema → registered server-side",
        "Python method on agent class → runs in-process",
    ],
    "Handle messages": [
        "threads_client.create_message() + runs_client.create_run()",
        "async def on_message(self, context, message)",
    ],
    "Manage state": [
        "Foundry thread object (automatic)",
        "context.session or custom data structure (explicit)",
    ],
    "Deploy": [
        "Agent is a Foundry resource — managed hosting",
        "Container image → Foundry hosted compute",
    ],
}

print(f"{'Aspect':<22} {'Path A: Prompt Agent':<52} {'Path B: MAF'}")
print("─" * 120)
for aspect, (path_a, path_b) in comparison.items():
    print(f"{aspect:<22} {path_a:<52} {path_b}")

---
## 7 · Cleanup

In [ ]:
# No server-side resources to clean up — the MAF agent is a local Python object.
# In a deployed scenario, you'd stop the container via the Foundry API.

del advisor
del conversation_history
print("✅ Local resources cleaned up")

---

<div style="background: linear-gradient(135deg, #0f3460, #533483); padding: 20px 25px; border-radius: 10px; color: white; margin-top: 25px;">

## ✅ Lab 02 Summary

We built a `ZavaBankAdvisor` class that handles the same advisory workflow as Path A, but with full Python control.

**What MAF gives you:**
- **`BaseAgent`** — a clean contract: implement `on_message`, get a production agent
- **In-process tools** — your methods, your data access, your error handling
- **Explicit state** — you decide what context the model sees at each turn
- **Python-native** — standard classes, standard testing, standard deployment

The agent is your code — you control every aspect of message handling, tool dispatch, and conversation management.

</div>